<a href="https://colab.research.google.com/github/avocado-planet/07-Context-Engineering/blob/main/07_context_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain コンテキストエンジニアリング

エージェントが失敗する最大の原因は「正しいコンテキストをLLMに渡していない」こと。
本ノートブックでは、LangChain の3つのコンテキスト制御手法を実践的に学びます。

## 目次

1. セットアップ
2. コンテキストエンジニアリングとは

**Part 1: Model Context（モデルコンテキスト）**
3. 動的システムプロンプト（`@dynamic_prompt`）
4. メッセージの注入（`@wrap_model_call`）
5. 動的ツール選択
6. 動的モデル切替
7. 動的レスポンスフォーマット

**Part 2: Tool Context（ツールコンテキスト）**
8. ツールからの読み取り（State / Store / Runtime Context）
9. ツールからの書き込み

**Part 3: Life-cycle Context（ライフサイクルコンテキスト）**
10. ガードレール（`@after_model`）
11. 自動要約（SummarizationMiddleware）
12. ロギング・監視
13. 総合例 — 全コンテキスト制御を組み合わせる

---
## 1. セットアップ

In [1]:
!pip install --pre -U langchain langgraph langchain-openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.1/173.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.4 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

MODEL = "openai:gpt-4.1-mini"
MODEL_CHEAP = "openai:gpt-4.1-nano"
print(f"モデル: {MODEL}")

モデル: openai:gpt-4.1-mini


In [3]:
# 共通ツール定義
from langchain_core.tools import tool
import random

@tool
def get_weather(city: str) -> str:
    """都市の天気を取得します"""
    return f"{city}: 晴れ, {random.randint(15,30)}度"

@tool
def search_orders(user_id: str, status: str) -> str:
    """ユーザーの注文を検索します。statusはpending/shipped/deliveredのいずれか。"""
    return f"ユーザー{user_id}の{status}注文: 3件見つかりました"

@tool
def delete_order(order_id: str) -> str:
    """注文を削除します（危険な操作）"""
    return f"注文{order_id}を削除しました"

@tool
def read_data(query: str) -> str:
    """データを読み取ります（安全な操作）"""
    return f"クエリ'{query}'の結果: サンプルデータ"

print("ツール定義完了")

ツール定義完了


---
## 2. コンテキストエンジニアリングとは

Andrej Karpathy の言葉:
> コンテキストエンジニアリングとは「次のステップに必要な正しい情報で
> コンテキストウィンドウを満たす繊細な技術と科学」

### 3つのコンテキスト制御

```
┌─────────────────────────────────────────────┐
│ Model Context（一時的）                       │
│ LLM の1回の呼び出しに何を渡すか              │
│ → プロンプト / メッセージ / ツール / モデル    │
├─────────────────────────────────────────────┤
│ Tool Context（永続的）                        │
│ ツールが何にアクセスし、何を生成するか        │
│ → State / Store / Runtime Context の読み書き  │
├─────────────────────────────────────────────┤
│ Life-cycle Context（永続的）                  │
│ モデル呼び出しとツール実行の「間」に何をするか│
│ → 要約 / ガードレール / ロギング              │
└─────────────────────────────────────────────┘
```

### 3つのデータソース

| データソース | 別名 | スコープ | 例 |
|---|---|---|---|
| **Runtime Context** | 静的設定 | 会話スコープ | user_id, APIキー, 権限 |
| **State** | 短期メモリ | 会話スコープ | メッセージ履歴, ツール結果 |
| **Store** | 長期メモリ | 会話横断 | ユーザー設定, 学習した知識 |

---
# Part 1: Model Context（モデルコンテキスト）

LLM の1回の呼び出しに何を渡すかを制御します。
**一時的（Transient）**な変更で、State には保存されません。

## 3. 動的システムプロンプト

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.messages import HumanMessage
from dataclasses import dataclass


# Runtime Context: invoke 時に渡す追加情報
@dataclass
class AppContext:
    user_role: str  # "admin" or "viewer"
    user_name: str


# ============================================
# State, Store, Runtime Context の全てを使って
# 動的にプロンプトを構築する
# ============================================
@dynamic_prompt
def smart_prompt(request: ModelRequest) -> str:
    # --- Runtime Context から読む ---
    role = request.runtime.context.user_role
    name = request.runtime.context.user_name

    # --- State から読む ---
    msg_count = len(request.messages)

    # 基本プロンプト
    base = f"あなたは親切なアシスタントです。ユーザー名: {name}。"

    # ロールに応じた指示
    if role == "admin":
        base += "\n管理者権限があります。全ての操作が可能です。"
    else:
        base += "\n閲覧のみ可能です。データの変更はできません。"

    # 会話が長くなったら簡潔に
    if msg_count > 10:
        base += "\n会話が長くなっています。簡潔に回答してください。"

    print(f"  [dynamic_prompt] role={role}, msgs={msg_count}")
    return base


agent_prompt = create_agent(
    model=MODEL,
    tools=[get_weather],
    middleware=[smart_prompt],
    context_schema=AppContext,
)

print("=== admin ユーザー ===")
r = agent_prompt.invoke(
    {"messages": [HumanMessage("こんにちは、私は何ができますか？")]},
    context=AppContext(user_role="admin", user_name="タロウ"),
)
print(f"応答: {r['messages'][-1].content[:120]}")

print("\n=== viewer ユーザー ===")
r = agent_prompt.invoke(
    {"messages": [HumanMessage("こんにちは、私は何ができますか？")]},
    context=AppContext(user_role="viewer", user_name="ハナコ"),
)
print(f"応答: {r['messages'][-1].content[:120]}")

print("\n=== viewer ユーザー ===")
r = agent_prompt.invoke(
    {"messages": [HumanMessage("注文を削除してください。")]},
    context=AppContext(user_role="viewer", user_name="ハナコ"),
)
print(f"応答: {r['messages'][-1].content[:120]}")

=== admin ユーザー ===
  [dynamic_prompt] role=admin, msgs=1
応答: こんにちは、タロウさん！私は親切なアシスタントです。あなたは管理者権限を持っているので、様々な操作や情報取得が可能です。例えば：

- 都市の天気を調べることができます。
- さまざまな質問に答えたり、情報提供ができます。
- ファイルやデ

=== viewer ユーザー ===
  [dynamic_prompt] role=viewer, msgs=1
応答: こんにちは、ハナコさん！私はあなたの質問に答えたり、情報を提供したり、お手伝いしたりできます。例えば、天気の情報を調べたり、日常生活のアドバイスをしたりすることができます。何か知りたいことや助けが必要なことがあれば教えてくださいね。

=== viewer ユーザー ===
  [dynamic_prompt] role=viewer, msgs=1
応答: 申し訳ありませんが、注文の削除はできません。ほかにお手伝いできることがあれば教えてください。


## 4. メッセージの注入

`@wrap_model_call` でモデルに渡すメッセージを一時的に加工します。
State は変更されません（Transient）。

In [7]:
from langchain.agents.middleware import wrap_model_call, ModelResponse
from typing import Callable


@wrap_model_call
def inject_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """モデルに渡すメッセージに追加情報を注入する（一時的）。"""
    # 追加コンテキストをメッセージ末尾に注入
    # ※ モデルは末尾のメッセージにより注意を払う傾向がある
    extra_info = {
        "role": "user",
        "content": (
            "【システム情報】\n"
            "- 現在の地域: 日本\n"
            "- 気温の単位: 摂氏\n"
            "- 言語: 日本語で回答してください"
        ),
    }
    messages = [*request.messages, extra_info]
    request = request.override(messages=messages)

    print("  [inject_context] 追加コンテキストを注入")
    return handler(request)


agent_inject = create_agent(
    model=MODEL,
    tools=[get_weather],
    middleware=[inject_context],
)

r = agent_inject.invoke({"messages": [HumanMessage("東京の天気は？")]})
print(f"応答: {r['messages'][-1].content}")

  [inject_context] 追加コンテキストを注入
  [inject_context] 追加コンテキストを注入
応答: 東京の現在の天気は晴れで、気温は20度です。


## 5. 動的ツール選択

ユーザーの権限に応じて、使えるツールを制限します。

In [13]:
@wrap_model_call
def filter_tools_by_role(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Runtime Context のロールに応じてツールをフィルタ。"""
    role = request.runtime.context.user_role

    if role == "admin":
        # admin は全ツール使える
        filtered = request.tools
    elif role == "editor":
        # editor は削除以外
        filtered = [t for t in request.tools if t.name != "delete_order"]
    else:
        # viewer は読み取りのみ
        filtered = [t for t in request.tools if t.name in ["read_data", "get_weather"]]

    print(f"  [filter_tools] role={role} → {[t.name for t in filtered]}")
    return handler(request.override(tools=filtered))


agent_tools = create_agent(
    model=MODEL,
    tools=[get_weather, search_orders, delete_order, read_data],
    middleware=[filter_tools_by_role],
    context_schema=AppContext,
)

print("=== viewer は read_data と get_weather のみ ===")
r = agent_tools.invoke(
    #{"messages": [HumanMessage("天気を調べて")]},
    #残念ですが、文言の表現を若干変えると、注文を削除できると判断するケースがある。。。
    {"messages": [HumanMessage("注文の削除をお願いします。order_id = 123")]},
    context=AppContext(user_role="viewer", user_name="ゲスト"),
)
print(f"応答: {r['messages'][-1].content[:100]}")

=== viewer は read_data と get_weather のみ ===
  [filter_tools] role=viewer → ['get_weather', 'read_data']
応答: 申し訳ありませんが、注文の削除に関する直接の操作はできません。ご注文の削除を希望される場合は、利用されているサービスのカスタマーサポートにご連絡いただくか、該当の注文管理画面から手続きをお願いいたしま


## 6. 動的モデル切替

会話の複雑さやコスト要件に応じてモデルを切り替えます。

In [21]:
from langchain.chat_models import init_chat_model

model_standard = init_chat_model(MODEL)
model_cheap = init_chat_model(MODEL_CHEAP)


@wrap_model_call
def select_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """メッセージ数に応じてモデルを切り替え。"""
    #msg_count = len(request.messages)
    msg_count = len(request.state["messages"])
    print(request.state["messages"] is request.messages)

    if msg_count > 10:
        model = model_standard  # 長い会話 → 高性能
        print(f"  [select_model] 標準モデル (msgs={msg_count})")
    else:
        model = model_cheap     # 短い会話 → 安価
        print(f"  [select_model] 軽量モデル (msgs={msg_count})")

    return handler(request.override(model=model))


agent_model = create_agent(
    model=MODEL,
    tools=[get_weather],
    middleware=[select_model],
)

#r = agent_model.invoke({"messages": [HumanMessage("こんにちは")]})
#print(f"応答: {r['messages'][-1].content[:80]}")

from langgraph.graph import MessagesState
state: MessagesState = {"messages": []}
items = ['車', '飛行機', 'バイク', '自転車','蒸気機関車','電車']
for idx, i in enumerate(items):
    print(f"\n=== Round {idx+1} ===")
    state["messages"] += [HumanMessage(content=f"{i}はいくつかの車輪があるか、簡単に答えてください")]
    result = agent_model.invoke(state)
    state["messages"] = result["messages"]
    print(f'content: {result["messages"][-1].content}')


=== Round 1 ===
True
  [select_model] 軽量モデル (msgs=1)
content: 一般的に、車には4つの車輪があります。

=== Round 2 ===
True
  [select_model] 軽量モデル (msgs=3)
content: 飛行機には一般的に3つか4つの車輪があります。

=== Round 3 ===
True
  [select_model] 軽量モデル (msgs=5)
content: バイクには通常、2つの車輪があります。

=== Round 4 ===
True
  [select_model] 軽量モデル (msgs=7)
content: 自転車には通常、2つの車輪があります。

=== Round 5 ===
True
  [select_model] 軽量モデル (msgs=9)
content: 蒸気機関車には多くの場合、4つ以上の車輪があります。

=== Round 6 ===
True
  [select_model] 標準モデル (msgs=11)
content: 電車には通常、車両ごとに8つ以上の車輪があります。


## 7. 動的レスポンスフォーマット

構造化出力のスキーマを状況に応じて変更します。

In [26]:
from pydantic import BaseModel, Field


class SimpleAnswer(BaseModel):
    """短い回答"""
    answer: str = Field(description="簡潔な回答")


class DetailedAnswer(BaseModel):
    """詳細な回答"""
    answer: str = Field(description="詳しい回答")
    reasoning: str = Field(description="理由の説明")
    confidence: float = Field(description="確信度 0.0〜1.0")


@wrap_model_call
def select_format(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """会話の段階に応じてレスポンス形式を変更。"""
    msg_count = len(request.messages)

    if msg_count < 3:
        fmt = SimpleAnswer
        print("  [select_format] SimpleAnswer")
    else:
        fmt = DetailedAnswer
        print("  [select_format] DetailedAnswer")

    return handler(request.override(response_format=fmt))


agent_fmt = create_agent(
    model=MODEL,
    tools=[],
    middleware=[select_format],
)

#r = agent_fmt.invoke({"messages": [HumanMessage("Pythonとは？")]})
#print(f"応答: {r['messages'][-1].content[:200]}")

from langgraph.graph import MessagesState
import json

state: MessagesState = {"messages": []}
items = ['車', '飛行機', 'バイク', '自転車', '蒸気機関車', '電車']
for idx, i in enumerate(items):
    print(f"\n=== Round {idx+1} ===")
    state["messages"] += [HumanMessage(content=f"{i}はいくつかの車輪があるか、簡単に答えてください")]
    result = agent_fmt.invoke(state)

    # 最後のAIMessageのcontentをJSONからテキストに正規化して履歴に戻す
    messages = result["messages"]
    last_msg = messages[-1]
    try:
        parsed = json.loads(last_msg.content)
        # answerフィールドだけ取り出してプレーンテキスト化
        plain = parsed.get("answer", last_msg.content)
        last_msg = last_msg.model_copy(update={"content": plain})
        messages = messages[:-1] + [last_msg]
    except (json.JSONDecodeError, AttributeError):
        pass

    state["messages"] = messages

    if parsed and "reasoning" in parsed:
        print(f'answer: {parsed["answer"]}')
        print(f'reasoning: {parsed["reasoning"]}')
        print(f'confidence: {parsed["confidence"]}')
    else:
        print(f'content: {last_msg.content}')


=== Round 1 ===
  [select_format] SimpleAnswer
content: 車には通常4つの車輪があります。

=== Round 2 ===
  [select_format] DetailedAnswer
answer: 飛行機の車輪の数は機種によって異なりますが、一般的な民間旅客機では通常6つから16つの車輪があります。
reasoning: 飛行機の車輪は着陸装置に配置されていて、機種によって車輪の数が異なります。小型の飛行機では少ない車輪数で済みますが、大型の旅客機では重さを支えるために多くの車輪が必要です。
confidence: 0.9

=== Round 3 ===
  [select_format] DetailedAnswer
answer: バイクには通常2つの車輪があります。
reasoning: バイクは二輪車として設計されており、前後に各1つずつ計2つの車輪が基本です。
confidence: 1.0

=== Round 4 ===
  [select_format] DetailedAnswer
answer: 自転車には通常2つの車輪があります。
reasoning: 一般的な自転車は前輪と後輪の2つの車輪で構成されているためです。三輪の自転車など例外もありますが、基本的には2つの車輪が標準です。
confidence: 1.0

=== Round 5 ===
  [select_format] DetailedAnswer
answer: 蒸気機関車の車輪の数は種類によって異なりますが、一般的には6つ以上の車輪があります。
reasoning: 蒸気機関車は動輪と従輪の組み合わせで構成されており、車両の種類や大きさによって車輪の数が大きく異なります。小型のものは6つ程度ですが、大型のものはさらに多くの車輪を持っています。
confidence: 0.9

=== Round 6 ===
  [select_format] DetailedAnswer
answer: 電車の車輪の数は車両の種類によって異なりますが、一般的に1両あたり8つの車輪（4つの台車に2つずつの車輪）があることが多いです。
reasoning: 電車は複数の台車（車輪が取り付けられている枠）を持っており、1台車に通常2

---
# Part 2: Tool Context（ツールコンテキスト）

ツールが何にアクセスし、何を生成するかを制御します。
ツールは State / Store / Runtime Context の**読み書き**ができます。

## 8. ツールからの読み取り

In [34]:
from langchain.tools import ToolRuntime
from langgraph.store.memory import InMemoryStore


@dataclass
class UserContext:
    user_id: str
    api_key: str


# --- Runtime Context から読む ---
@tool
def fetch_user_data(query: str, runtime: ToolRuntime[UserContext]) -> str:
    """ユーザーデータを取得。runtime.context からuser_idを取得。"""
    user_id = runtime.context.user_id
    return f"ユーザー{user_id}のデータ: クエリ'{query}'の結果"


# --- Store（長期メモリ）から読む ---
@tool
def get_preferences(runtime: ToolRuntime[UserContext]) -> str:
    """保存済みのユーザー設定を取得。"""
    assert runtime.store is not None
    user_id = runtime.context.user_id
    prefs = runtime.store.get(("preferences",), user_id)
    if prefs:
        return f"設定: {prefs.value}"
    return "設定が保存されていません"


# Store にテスト用データを事前投入
store = InMemoryStore()
store.put(("preferences",), "user_001", {
    "language": "ja",
    "theme": "dark",
    "notification": True,
})

agent_tool_read = create_agent(
    model=MODEL,
    tools=[fetch_user_data, get_preferences],
    store=store,
    context_schema=UserContext,
)

print("=== ツールから Runtime Context + Store を読む ===")
r = agent_tool_read.invoke(
    {"messages": [HumanMessage("私の設定と、ユーザーデータ見せて")]},
    context=UserContext(user_id="user_001", api_key="sk-xxx"),
)
print(f"応答: {r['messages'][-1].content[:200]}")

=== ツールから Runtime Context + Store を読む ===
応答: あなたの設定は次の通りです：
- 言語: 日本語
- テーマ: ダーク
- 通知: 有効

ユーザーデータについては、クエリに対する結果を取得しましたが、具体的な内容がありませんでした。もし特定の情報を知りたい場合は教えてください。


フローの全体像


```
invoke(context=UserContext(user_id="user_001"))
        │
        ▼
   LLM が判断
   「fetch_user_dataとget_preferencesを呼ぼう」
        │
        ├─→ fetch_user_data(query="...")
        │       runtime.context.user_id → "user_001"  ← invokeから
        │
        └─→ get_preferences()
                runtime.store.get(("preferences",), "user_001")
                → {language:"ja", theme:"dark"...}  ← storeから
```



## 9. ツールからの書き込み

ツールの実行結果を Store に永続保存したり、
State を更新して後続のモデル呼び出しに影響を与えます。

In [37]:
from typing_extensions import TypedDict


class PreferenceData(TypedDict):
    key: str
    value: str


@tool
def save_preference(
    preference: PreferenceData,
    runtime: ToolRuntime[UserContext],
) -> str:
    """ユーザー設定を保存します。keyとvalueを指定してください。"""
    assert runtime.store is not None
    user_id = runtime.context.user_id

    # 既存の設定を取得
    existing = runtime.store.get(("preferences",), user_id)
    data = existing.value if existing else {}

    # 更新して保存
    data[preference["key"]] = preference["value"]
    runtime.store.put(("preferences",), user_id, data)

    return f"設定を保存: {preference['key']}={preference['value']}"


store_rw = InMemoryStore()

agent_tool_write = create_agent(
    model=MODEL,
    tools=[save_preference, get_preferences],
    store=store_rw,
    context_schema=UserContext,
)

print("=== ツールから Store に書き込み ===")
r = agent_tool_write.invoke(
    {"messages": [HumanMessage("テーマをライトモードに設定して")]},
    context=UserContext(user_id="user_t", api_key="sk-xxx"),
)
print(f"応答: {r['messages'][-1].content[:150]}")

# Store の内容を直接確認
stored = store_rw.get(("preferences",), "user_t")
print(f"\nStore の内容: {stored.value if stored else 'なし'}")

=== ツールから Store に書き込み ===
応答: テーマをライトモードに設定しました。ほかに何かご希望はありますか？

Store の内容: {'theme': 'light'}


---
# Part 3: Life-cycle Context（ライフサイクルコンテキスト）

モデル呼び出しとツール実行の **間** に何をするかを制御します。
State に**永続的**な変更を加えます。

## 10. ガードレール（`@after_model`）

In [38]:
from langchain.agents.middleware import after_model, AgentState, hook_config
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any


class GuardrailMiddleware:
    """モデル応答のガードレール。
    NGワードが含まれていたらブロックする。
    """

@after_model
@hook_config(can_jump_to=["end"])
def content_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """モデル応答をチェックし、不適切な内容をブロック。"""
    last = state["messages"][-1]
    content = str(last.content).lower()

    BLOCKED_WORDS = ["password", "secret", "credential", "パスワード"]

    for word in BLOCKED_WORDS:
        if word in content:
            print(f"  [guardrail] ブロック! NGワード: '{word}'")
            return {
                "messages": [AIMessage(
                    "申し訳ございません。セキュリティ上の理由により、"
                    "その情報は提供できません。"
                )],
                "jump_to": "end",
            }

    print("  [guardrail] OK")
    return None


agent_guard = create_agent(
    model=MODEL,
    tools=[],
    middleware=[content_guardrail],
)

print("=== ガードレール ===")
r = agent_guard.invoke({"messages": [HumanMessage("パスワードを変更したい")]})
print(f"通常応答: {r['messages'][-1].content[:80]}")

=== ガードレール ===
  [guardrail] ブロック! NGワード: 'パスワード'
通常応答: 申し訳ございません。セキュリティ上の理由により、その情報は提供できません。


## 11. 自動要約（SummarizationMiddleware）

In [40]:
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver


agent_sum = create_agent(
    model=MODEL,
    tools=[get_weather],
    middleware=[
        SummarizationMiddleware(
            model=MODEL_CHEAP,
            trigger=("tokens", 2000),
            keep=("messages", 5),
        ),
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "sum-demo"}}

msgs = [
    "私はタロウです。東京在住のエンジニアです。",
    "趣味はPythonと登山です。",
    "最近LangChainを勉強しています。",
    "好きな食べ物はラーメンです。",
    "ペットは猫を飼っています。",
    "私について覚えていることを教えて。",
]

print("=== 自動要約 ===")
for msg in msgs:
    r = agent_sum.invoke(
        {"messages": [HumanMessage(msg)]}, config=config
    )
    print(f"[{len(r['messages'])}件] {msg[:30]} → {r['messages'][-1].content[:60]}")

=== 自動要約 ===
[2件] 私はタロウです。東京在住のエンジニアです。 → タロウさん、こんにちは！東京でエンジニアをされているんですね。何かお手伝いできることがあれば、気軽に教えてくださいね。
[4件] 趣味はPythonと登山です。 → Pythonと登山が趣味なんて素敵ですね！Pythonで何か特に興味のあるプロジェクトや分野はありますか？また、最近登っ
[6件] 最近LangChainを勉強しています。 → LangChainの勉強をされているんですね！LangChainは言語モデルを活用したアプリケーション開発にとても役立つ
[8件] 好きな食べ物はラーメンです。 → ラーメンがお好きなんですね！東京には本当にたくさんの美味しいラーメン店がありますよね。特に好きなラーメンの種類や、おすす
[10件] ペットは猫を飼っています。 → 猫ちゃんを飼っているんですね！猫はとても癒されますよね。猫の名前や性格、好きな遊びなどがあればぜひ教えてください。猫に関
[12件] 私について覚えていることを教えて。 → もちろんです！タロウさんについて覚えていることをお伝えしますね。

- 東京在住のエンジニア
- 趣味はPythonと登


## 12. ロギング・監視

In [41]:
from langchain.agents.middleware import (
    AgentMiddleware, before_agent, before_model, after_model, after_agent,
    wrap_model_call, wrap_tool_call,
)
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import ToolMessage
from langgraph.types import Command
import time
import json


class ObservabilityMiddleware(AgentMiddleware):
    """全ライフサイクルイベントを監視するミドルウェア。"""

    def __init__(self):
        super().__init__()
        self.start_time = None
        self.model_calls = 0
        self.tool_calls = 0

    def before_agent(self, state, runtime):
        self.start_time = time.time()
        self.model_calls = 0
        self.tool_calls = 0
        print("📋 [lifecycle] agent_start")
        return None

    def before_model(self, state, runtime):
        self.model_calls += 1
        print(f"📋 [lifecycle] before_model #{self.model_calls}")
        return None

    def after_model(self, state, runtime):
        last = state["messages"][-1]
        has_tools = hasattr(last, "tool_calls") and bool(last.tool_calls)
        print(f"📋 [lifecycle] after_model (has_tools={has_tools})")
        return None

    def wrap_tool_call(self, request, handler):
        self.tool_calls += 1
        name = request.tool_call["name"]
        start = time.time()
        result = handler(request)
        elapsed = time.time() - start
        print(f"📋 [lifecycle] tool '{name}' ({elapsed:.3f}s)")
        return result

    def after_agent(self, state, runtime):
        elapsed = time.time() - self.start_time if self.start_time else 0
        print(f"📋 [lifecycle] agent_end: {self.model_calls}回モデル, {self.tool_calls}回ツール, {elapsed:.2f}秒")
        return None


agent_obs = create_agent(
    model=MODEL,
    tools=[get_weather],
    middleware=[ObservabilityMiddleware()],
)

print("=== ライフサイクル監視 ===")
r = agent_obs.invoke({"messages": [HumanMessage("東京の天気を教えて")]})
print(f"\n応答: {r['messages'][-1].content[:80]}")

=== ライフサイクル監視 ===
📋 [lifecycle] agent_start
📋 [lifecycle] before_model #1
📋 [lifecycle] after_model (has_tools=True)
📋 [lifecycle] tool 'get_weather' (0.001s)
📋 [lifecycle] before_model #2
📋 [lifecycle] after_model (has_tools=False)
📋 [lifecycle] agent_end: 2回モデル, 1回ツール, 1.01秒

応答: 現在の東京の天気は晴れで、気温は16度です。ほかに知りたいことはありますか？


## 13. 総合例 — 全コンテキスト制御を組み合わせる

In [42]:
production_agent = create_agent(
    model=MODEL,
    tools=[get_weather, read_data, search_orders, delete_order],
    middleware=[
        # Life-cycle: 全体監視
        ObservabilityMiddleware(),
        # Model Context: 動的プロンプト
        smart_prompt,
        # Model Context: ツールフィルタ
        filter_tools_by_role,
        # Life-cycle: ガードレール
        content_guardrail,
    ],
    context_schema=AppContext,
)

print("=" * 60)
print("総合例: 全コンテキスト制御")
print("=" * 60)
r = production_agent.invoke(
    {"messages": [HumanMessage("東京の天気と、注文の状況を教えて")]},
    context=AppContext(user_role="viewer", user_name="ゲスト"),
)
print(f"\n最終応答: {r['messages'][-1].content[:200]}")

総合例: 全コンテキスト制御
📋 [lifecycle] agent_start
📋 [lifecycle] before_model #1
  [dynamic_prompt] role=viewer, msgs=1
  [filter_tools] role=viewer → ['get_weather', 'read_data']
  [guardrail] OK
📋 [lifecycle] after_model (has_tools=True)
📋 [lifecycle] tool 'get_weather' (0.001s)
📋 [lifecycle] tool 'read_data' (0.001s)
📋 [lifecycle] before_model #2
  [dynamic_prompt] role=viewer, msgs=4
  [filter_tools] role=viewer → ['get_weather', 'read_data']
  [guardrail] OK
📋 [lifecycle] after_model (has_tools=False)
📋 [lifecycle] agent_end: 2回モデル, 2回ツール, 1.82秒

最終応答: 現在の東京の天気は晴れで、気温は16度です。
また、注文の状況については「サンプルデータ」という結果が得られています。具体的な内容の詳細が必要であればお知らせください。


---
## まとめ

| コンテキスト | 永続性 | 手段 | 例 |
|---|---|---|---|
| **Model** | 一時的 | `@dynamic_prompt`, `@wrap_model_call` | プロンプト, ツール選択, モデル切替 |
| **Tool** | 永続的 | `ToolRuntime` で読み書き | State/Store/Context アクセス |
| **Life-cycle** | 永続的 | `@before_model`, `@after_model` 等 | 要約, ガードレール, ロギング |

### データソース

| ソース | スコープ | アクセス方法 |
|---|---|---|
| **Runtime Context** | 会話スコープ | `request.runtime.context`, `runtime.context` |
| **State** | 会話スコープ | `request.state`, `state["messages"]` |
| **Store** | 会話横断 | `request.runtime.store`, `runtime.store` |